# Data Cleaning - Diseases and Symptoms Dataset

**Input:** `Dataset/Data/diseases_and_symptoms.csv`

**Output:** `Dataset/Clean_Data/diseases_and_symptoms_clean.csv`

---

## Vấn đề phát hiện từ EDA (II_diseases_and_symptoms.ipynb)

| # | Vấn đề | Mô tả | Mức độ ảnh hưởng |
|---|--------|-------|------------------|
| 1 | Trùng lặp | 57,298 hàng trùng lặp hoàn toàn (~23.2%) | Cao - làm tăng bias, giảm hiệu quả huấn luyện |
| 2 | Kích thước file | ~190MB, 246,945 hàng × 378 cột | Trung bình - cần tối ưu memory |
| 3 | Kiểu dữ liệu | CSV đọc mặc định int64 (8 bytes/giá trị) | Trung bình - có thể giảm xuống bool (1 byte) |
| 4 | Missing values | 0 - không có giá trị thiếu | Thấp |
| 5 | Giá trị vô lý | 0 - tất cả triệu chứng nằm trong {0,1} | Thấp |
| 6 | Số lớp bệnh | 773 lớp, phân bổ khá đồng đều (~1,200 mẫu/lớp) | Trung bình - cân nhắc cho multi-class |
1. **Memory optimization:** Đọc CSV với dtype=category cho diseases và dtype=bool cho symptoms, giảm memory từ ~500MB xuống ~90MB.
2. **Duplicate removal:** Loại bỏ 57,298 hàng trùng lặp hoàn toàn bằng drop_duplicates().
3. **Data validation:** Xác nhận không có missing values, không có giá trị ngoài {0,1}.
4. **Label preservation:** Giữ nguyên cột diseases dạng category để preserve 773 lớp.
5. **Export:** Lưu kết quả ra Clean_Data/diseases_and_symptoms_clean.csv.

---


In [ ]:
import pandas as pd
import numpy as np
import os
import time

# ============================================
# CONFIGURATION
# ============================================
INPUT_PATH = r"D:\PYTHON\Dataset\Data\diseases_and_symptoms.csv"
OUTPUT_DIR = r"D:\PYTHON\Dataset\Clean_Data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "diseases_and_symptoms_clean.csv")

def timeit(msg):
    t0 = time.time()
    print(f'[START] {msg}')
    return t0

def elapsed(t0, msg):
    t1 = time.time()
    print(f'[DONE]  {msg} — {t1-t0:.2f}s')
    return t1-t0


In [ ]:
t0 = timeit('Reading raw CSV...')

# Read only header first to get column names
df_raw = pd.read_csv(INPUT_PATH, nrows=0)
symptom_cols = [c for c in df_raw.columns if c != "diseases"]
dtype_map = {'diseases': 'category'}
for c in symptom_cols:
    dtype_map[c] = 'bool'

# Read full CSV with optimized dtypes
df = pd.read_csv(INPUT_PATH, dtype=dtype_map)
elapsed(t0, f'Read CSV: {df.shape[0]:,} rows × {df.shape[1]} cols')

print('\nColumns (first 10):', df.columns.tolist()[:10], '...')
print('Data types (first 10):')
print(df.dtypes.head(10))

print('\nMemory usage:')
print(f"{df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print('\nPreview (first 5 rows, first 12 columns):')
df.iloc[:5, :12]


In [ ]:
t0 = timeit('Checking duplicates...')
dup_count = df.duplicated().sum()
elapsed(t0, f'Duplicate check: {dup_count:,} rows')

print(f'\nTỷ lệ trùng lặp: {dup_count / len(df) * 100:.1f}%')

if dup_count > 0:
    print('\nMẫu hàng trùng lặp (chỉ 5 dòng, 10 cột đầu):')
    dup_mask = df.duplicated(keep=False)
    print(df.loc[dup_mask, df.columns[:10]].head())


In [ ]:
t0 = timeit('Removing duplicates...')
rows_before = len(df)
df_clean = df.drop_duplicates().reset_index(drop=True)
rows_after = len(df_clean)
elapsed(t0, f'Removed {rows_before - rows_after:,} duplicates')

print(f'\n=== Trước khi xóa trùng lặp ===')
print(f"Số dòng: {rows_before:,}")

print(f'\n=== Sau khi xóa trùng lặp ===')
print(f"Số dòng: {rows_after:,}")
print(f"Đã xóa: {rows_before - rows_after:,} dòng ({(rows_before-rows_after)/rows_before*100:.1f}%)")


In [ ]:
print('=== VALIDATION ===')

# Check missing values
missing_sum = df_clean.isnull().sum().sum()
print(f'\nMissing values: {missing_sum}')

# Check invalid values (outside {0, 1})
invalid = (~df_clean[symptom_cols].isin([False, True])).sum().sum()
print(f'Invalid values (outside {0,1}): {invalid}')

# Check duplicates again
dup_final = df_clean.duplicated().sum()
print(f'Remaining duplicates: {dup_final}')

# Label distribution
label_counts = df_clean["diseases"].value_counts()
print(f'\nSố lớp bệnh: {label_counts.shape[0]}')
print('Top 10 lớp bệnh:')
print(label_counts.head(10))

print('\n=== Validation Complete ===')


In [ ]:
t0 = timeit('Exporting cleaned data...')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Convert bool back to int to keep CSV compact (0/1 instead of True/False)
df_export = df_clean.copy()
for c in symptom_cols:
    df_export[c] = df_export[c].astype(int)

df_export.to_csv(OUTPUT_FILE, index=False)
elapsed(t0, 'Export completed')

print(f"\nCleaned data exported to: D:\\PYTHON\\Dataset\\Clean_Data\\diseases_and_symptoms_clean.csv")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / 1024**2:.1f} MB")
print(f"Shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} cols")


| Chỉ tiêu | Trước | Sau |
|----------|-------|-----|
| Số dòng | 246,945 | 189,647 |
| Số cột | 378 | 378 |
| Trùng lặp | 57,298 (~23.2%) | 0 |
| Missing values | 0 | 0 |
| Giá trị vô lý | 0 | 0 |
| Số lớp bệnh | 773 | 773 |
| Memory (in-RAM) | ~500MB | ~90MB |
| Output | - | `Clean_Data/diseases_and_symptoms_clean.csv` |

---
- Đã loại bỏ **57,298 hàng trùng lặp** (~23.2% tổng dữ liệu).
- Giữ nguyên cấu trúc dữ liệu: 1 cột nhãn `diseases` (category) + 377 cột triệu chứng (bool).
- Không có missing values hay giá trị vô lý cần xử lý thêm.
- Dữ liệu sạch đã sẵn sàng cho bước chuẩn hóa, cân bằng lớp, và huấn luyện mô hình.
